## Tungsten — whole-stage codegen

Catalyst gives you the right *plan*. **Tungsten** makes that plan run fast on the JVM. Two ideas do most of the work:

- **Off-heap memory management** — Spark allocates structured binary buffers *off* the JVM heap, bypassing the garbage collector. Less GC pause, and cache-friendlier memory layouts.
- **Whole-stage code generation** — for each stage, Catalyst generates a *single* tightly-fused JVM method that pushes a row through every operator at once (filter, then project, then aggregate). One loop, instead of one method call per operator per row.

This is the deep reason DataFrames beat RDDs on structured data: an RDD has no schema, so Tungsten can't lay it out efficiently or generate per-stage code. With a DataFrame, the schema is a contract Tungsten compiles against.

You can see the proof in a physical plan: DataFrame operators are wrapped in a `*(N) WholeStageCodegen` block, where `N` is the codegen stage id.